# Extract wav2vec2 embeddings for the fixed-layer ablation (Colab GPU)

Layers **3, 6, 12, 18, 24** from the frozen distorted wavs, one forward pass per file, one
archive per layer saved to Drive. Pre-registered plan: `docs/ablation_plan.md`.
Layer 9 already exists, as `cv27_embeddings_layer09.tar`.

**Layer index convention.** A layer is an index into `outputs.hidden_states` of
`Wav2Vec2Model(..., output_hidden_states=True)`:
- 0 is the feature projection plus the positional embedding.
- k is the output of transformer block k.
- 24 is block 24 after the encoder's final LayerNorm.

This is the same code path that produced layer 9 (`src/preprocessing/extract_embedding.py`).
Before extracting, the notebook recomputes layer 9 for a sample of files and compares it
with the existing layer-9 archive.

**Needs in `DRIVE_DATA`:**
- `cv27_distorted_wavs.tar` and `cv27_distorted_wavs.tar.sha256`, both built on the Mac
- `cv27_embeddings_layer09.tar` and `SHA256SUMS`, already there from the layer-9 runs

**Writes to `DRIVE_DATA`:**
- `cv27_embeddings_layer<NN>.tar` and `cv27_embeddings_layer<NN>.tar.sha256`, about 3.5 GB per layer
- `cv27_embeddings_layers_info.json`
- `cv27_embeddings_layers_norms.csv`

Layers whose archive is already on Drive are skipped. If the session restarts, run all
cells again: files already extracted on this runtime's local disk are not recomputed.

In [ ]:
# ---- settings -------------------------------------------------------------
LAYERS = [3, 6, 12, 18, 24]   # hidden_states indices (see above); drop some if Drive is short of space

REPO_URL = "https://github.com/KankaveeRamsri/thai-dsr.git"
REF = "w3-layer-ablation"     # branch, tag or commit sha

DRIVE_ROOT = "/content/drive/MyDrive/thai-dsr"
DRIVE_DATA = f"{DRIVE_ROOT}/cv27_data"
WAV_ARCHIVE = "cv27_distorted_wavs.tar"

WORK = "/content/cv27_extract"   # local disk: wavs, per-layer embeddings, archives
REPO_DIR = "/content/thai-dsr"
N_CHECK = 12                     # files compared with the existing layer-9 archive
CHECK_REL_TOL = 1e-3             # max relative L2 error allowed in that check

assert all(0 <= layer <= 24 for layer in LAYERS) and 9 not in LAYERS, LAYERS

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
GPU_NAME = torch.cuda.get_device_name(0)
# Full fp32, as on the CPU that produced layer 9: no TF32 in convolutions or matmuls (A100/L4).
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print("GPU:", GPU_NAME, "| torch", torch.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", REF], check=True)
branch = subprocess.run(["git", "-C", REPO_DIR, "symbolic-ref", "-q", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
if branch:
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--quiet", "--hard", f"origin/{branch}"], check=True)
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
os.chdir(REPO_DIR)
print("repo", REPO_DIR, "| ref", REF, "| commit", COMMIT)

In [ ]:
# torch is preinstalled on Colab and satisfies requirements.txt, so it is not reinstalled.
!pip install -q -r requirements.txt

In [ ]:
import hashlib
import shutil
import time

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 24), b""):
            digest.update(chunk)
    return digest.hexdigest()

def expected_sha(name):
    sidecar = f"{DRIVE_DATA}/{name}.sha256"
    if os.path.exists(sidecar):
        return open(sidecar).read().split()[0]
    sums = dict(reversed(line.split()) for line in open(f"{DRIVE_DATA}/SHA256SUMS") if line.strip())
    return sums[name]

# Layers already archived on Drive are skipped.
TODO = [layer for layer in LAYERS
        if not (os.path.exists(f"{DRIVE_DATA}/cv27_embeddings_layer{layer:02d}.tar")
                and os.path.exists(f"{DRIVE_DATA}/cv27_embeddings_layer{layer:02d}.tar.sha256"))]
print("to extract:", TODO, "| already on Drive:", sorted(set(LAYERS) - set(TODO)))

total, used, free = shutil.disk_usage("/content/drive/MyDrive")
need = 3.6e9 * len(TODO)
print(f"Drive: {free / 1e9:.1f} GB free, about {need / 1e9:.1f} GB needed for {len(TODO)} archive(s)")
if TODO and free < need:
    print("WARNING: not enough Drive space for all layers. Free space, or shorten LAYERS and run again.")

WAVS = f"{WORK}/distorted"
if TODO and not os.path.exists(f"{WAVS}/.extracted"):
    os.makedirs(WORK, exist_ok=True)
    start = time.time()
    shutil.copy(f"{DRIVE_DATA}/{WAV_ARCHIVE}", f"{WORK}/{WAV_ARCHIVE}")
    assert sha256(f"{WORK}/{WAV_ARCHIVE}") == expected_sha(WAV_ARCHIVE), "checksum mismatch: " + WAV_ARCHIVE
    subprocess.run(["tar", "-xf", f"{WORK}/{WAV_ARCHIVE}", "-C", WORK], check=True)
    os.remove(f"{WORK}/{WAV_ARCHIVE}")
    open(f"{WAVS}/.extracted", "w").close()
    print(f"copied + verified + unpacked {WAV_ARCHIVE} in {time.time() - start:.0f}s")

if TODO:
    counts = {sev: len([f for f in os.listdir(f"{WAVS}/{sev}") if f.endswith(".wav")])
              for sev in ("mild", "moderate", "severe")}
    print(counts)
    assert counts == {"mild": 308, "moderate": 308, "severe": 2996}, counts
    WAV_PATHS = sorted(f"{WAVS}/{sev}/{f}" for sev in counts for f in os.listdir(f"{WAVS}/{sev}")
                       if f.endswith(".wav"))
    # Every wav must be the frozen file the layer-9 embeddings came from.
    import pandas as pd
    params = pd.read_csv("data/manifests/cv27_th_distortion_params.csv", usecols=["distorted_path", "distorted_sha256"])
    bad = [p for p, h in zip(params["distorted_path"], params["distorted_sha256"]) if sha256(f"{WORK}/{p}") != h]
    assert len(params) == len(WAV_PATHS) and not bad, f"{len(bad)} wav(s) differ from the frozen set, e.g. {bad[:3]}"
    print(f"all {len(params)} wavs match the frozen checksums in data/manifests/cv27_th_distortion_params.csv")

In [ ]:
# Layer-9 check: the GPU path here must reproduce the existing layer-9 embeddings.
import random
import numpy as np
import transformers
from src.preprocessing.extract_embedding import extract_all_hidden_states, load_model

feature_extractor, model = load_model(torch.device("cuda"))
MODEL_REVISION = getattr(model.config, "_commit_hash", None)
print("model revision", MODEL_REVISION, "| transformers", transformers.__version__)
assert model.config.num_hidden_layers == 24 and model.config.do_stable_layer_norm

CHECK = []
if TODO:
    sample = random.Random(42).sample(WAV_PATHS, N_CHECK)
    members = [f"embeddings/{os.path.basename(p)[:-4]}_layer09.npy" for p in sample]
    ref_dir = f"{WORK}/layer09_check"
    os.makedirs(ref_dir, exist_ok=True)
    start = time.time()
    subprocess.run(["tar", "-xf", f"{DRIVE_DATA}/cv27_embeddings_layer09.tar", "-C", ref_dir, *members], check=True)
    print(f"read {len(members)} layer-9 files from the archive in {time.time() - start:.0f}s")
    for path, member in zip(sample, members):
        hidden = extract_all_hidden_states(path, feature_extractor, model, torch.device("cuda"))
        assert len(hidden) == 25
        new = hidden[9].squeeze(0).cpu().numpy()
        old = np.load(f"{ref_dir}/{member}")
        assert new.shape == old.shape, (path, new.shape, old.shape)
        rel = float(np.linalg.norm(new - old) / np.linalg.norm(old))
        CHECK.append({"file": os.path.basename(path), "frames": int(new.shape[0]), "rel_l2": rel,
                      "max_abs": float(np.abs(new - old).max())})
    worst = max(c["rel_l2"] for c in CHECK)
    print(f"layer 9 vs archive on {len(CHECK)} files: max relative L2 error {worst:.2e}")
    assert worst < CHECK_REL_TOL, CHECK

In [ ]:
# One forward pass per file; every layer in TODO is saved from it.
import csv
from tqdm.auto import tqdm
from src.preprocessing.extract_embedding import cache_exists, save_hidden_states

def layer_dir(layer):
    return f"{WORK}/layer{layer:02d}/embeddings"

NORMS = f"{WORK}/norms.csv"   # mean frame L2 norm per file and layer (descriptive, for the plan)
start = time.time()
if TODO:
    pending = [p for p in WAV_PATHS if not all(cache_exists(p, layer_dir(l), layer=l) for l in TODO)]
    print(f"{len(WAV_PATHS) - len(pending)} file(s) already done, {len(pending)} to go")
    with open(NORMS, "a", newline="") as handle:
        writer = csv.writer(handle)
        for path in tqdm(pending, mininterval=30):
            hidden = extract_all_hidden_states(path, feature_extractor, model, torch.device("cuda"))
            for layer in TODO:
                save_hidden_states(hidden, path, layer_dir(layer), layer=layer, verbose=False)
                writer.writerow((os.path.basename(path)[:-4], layer,
                                 float(hidden[layer].squeeze(0).norm(dim=-1).mean())))
            handle.flush()
    for layer in TODO:
        n = len([f for f in os.listdir(layer_dir(layer)) if f.endswith(f"_layer{layer:02d}.npy")])
        print(f"layer {layer:02d}: {n} files")
        assert n == 3612, (layer, n)
print(f"extraction done in {time.time() - start:.0f}s on {GPU_NAME}")

In [ ]:
# One archive per layer (same layout as cv27_embeddings_layer09.tar: embeddings/<clip>_<sev>_layerNN.npy).
ARCHIVES = {}
for layer in TODO:
    name = f"cv27_embeddings_layer{layer:02d}.tar"
    local = f"{WORK}/{name}"
    start = time.time()
    subprocess.run(["tar", "-cf", local, "-C", f"{WORK}/layer{layer:02d}", "embeddings"], check=True)
    digest = sha256(local)
    shutil.copy(local, f"{DRIVE_DATA}/{name}.partial")
    os.replace(f"{DRIVE_DATA}/{name}.partial", f"{DRIVE_DATA}/{name}")
    with open(f"{DRIVE_DATA}/{name}.sha256", "w") as handle:
        handle.write(f"{digest}  {name}\n")
    os.remove(local)
    ARCHIVES[name] = {"sha256": digest, "bytes": os.path.getsize(f"{DRIVE_DATA}/{name}")}
    print(f"{name}: {ARCHIVES[name]['bytes'] / 1e9:.2f} GB, sha256 {digest[:12]}..., {time.time() - start:.0f}s")

In [ ]:
import json
import pandas as pd

if TODO:
    norms = pd.read_csv(NORMS, names=["file", "layer", "mean_frame_norm"]).drop_duplicates(["file", "layer"])
    norms.to_csv(f"{DRIVE_DATA}/cv27_embeddings_layers_norms.csv", index=False)
    info_path = f"{DRIVE_DATA}/cv27_embeddings_layers_info.json"
    info = json.load(open(info_path)) if os.path.exists(info_path) else {"runs": []}
    info["runs"].append({
        "finished": time.strftime("%Y-%m-%d %H:%M:%S"), "layers": TODO, "commit": COMMIT, "gpu": GPU_NAME,
        "torch": torch.__version__, "transformers": transformers.__version__,
        "model": model.config._name_or_path, "model_revision": MODEL_REVISION, "tf32": False,
        "files": len(WAV_PATHS), "wav_archive_sha256": expected_sha(WAV_ARCHIVE),
        "layer9_check": {"n": len(CHECK), "max_rel_l2": max(c["rel_l2"] for c in CHECK), "files": CHECK},
        "archives": ARCHIVES,
    })
    json.dump(info, open(info_path, "w"), indent=2)
    print(norms.groupby("layer")["mean_frame_norm"].describe().round(1))
print("archives on Drive:", sorted(f for f in os.listdir(DRIVE_DATA) if f.startswith("cv27_embeddings_layer")))